# 📌 TAHAP 1: PENGATURAN LINGKUNGAN & KONEKSI GOOGLE DRIVE
Sel ini bertujuan untuk menghubungkan Google Colab dengan Google Drive serta menginstal seluruh pustaka Python yang dibutuhkan dalam proyek Computer Vision ini.


In [ ]:
# 1. Mount Google Drive & Install Pustaka
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-image scikit-learn pandas numpy opencv-python joblib matplotlib seaborn xgboost lightgbm imbalanced-learn


# 📌 TAHAP 2: PEMUATAN DATASET & EKSTRAKSI OTOMATIS
Sel ini berfungsi untuk mengekstrak file `datasetCV.zip` secara otomatis ke direktori Colab dan memuat file pelabelan `labels.csv`.


In [ ]:
import pandas as pd
import os
import zipfile

# 2. Ekstrak datasetCV.zip jika belum diekstrak
if not os.path.exists("/content/iam_words"):
    zip_path = "/content/drive/MyDrive/DatasetCV/datasetCV.zip"
    if os.path.exists(zip_path):
        print("📦 Mengekstrak datasetCV.zip dari Google Drive ke /content ...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("/content")
        print("✅ Ekstraksi selesai!")

# 3. Memuat Labels.csv
labels_path = "/content/drive/MyDrive/DatasetCV/labels.csv"

if os.path.exists(labels_path):
    try:
        df = pd.read_csv(labels_path)
        if "filename" not in df.columns or "label" not in df.columns:
            df = pd.read_csv(labels_path, names=["filename", "label"])
    except Exception as e:
        df = pd.read_csv(labels_path, names=["filename", "label"])
        
    print("
✅ Berhasil memuat labels.csv!")
    print("Total Data Terlabel:", len(df))
    print("
Distribusi Label Asli:")
    print(df["label"].value_counts())
    print("
5 Data Pertama:")
    print(df.head())
else:
    print("❌ File labels.csv tidak ditemukan di:", labels_path)


# 📌 TAHAP 3: PREPROCESSING CITRA CANGGIH & PEMBERSIHAN NOISE
Sel ini mengimplementasikan teknik pengolahan citra digital murni (Background Normalization, Adaptive Gaussian Thresholding, Connected Components Denoising, dan Tight Bounding Box Crop) untuk menghasilkan citra biner stroke tulisan yang sangat bersih.


In [ ]:
import cv2
import numpy as np

def preprocess_and_filter(path):
    actual_path = path
    if not os.path.exists(actual_path):
        filename = os.path.basename(path)
        possible_paths = [
            os.path.join("/content", filename),
            os.path.join("/content/iam_words", filename),
            os.path.join("/content/iam_words/iam_words/words", filename),
            os.path.join("/content/drive/MyDrive/DatasetCV", filename),
            os.path.join("/content/drive/MyDrive/DatasetCV/iam_words", filename)
        ]
        for p in possible_paths:
            if os.path.exists(p):
                actual_path = p
                break
                
        if not os.path.exists(actual_path):
            for root, dirs, files in os.walk("/content"):
                if filename in files:
                    actual_path = os.path.join(root, filename)
                    break

    img = cv2.imread(actual_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, 1.0
    
    ih, iw = img.shape
    if ih < 15 or iw < 15:
        return None, 1.0

    kernel_bg = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    bg = cv2.morphologyEx(img, cv2.MORPH_DILATE, kernel_bg)
    norm = cv2.divide(img, bg, scale=255)
    
    binary = cv2.adaptiveThreshold(norm, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 10)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
    clean_binary = np.zeros_like(binary)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 8:
            clean_binary[labels == i] = 255
            
    pts = cv2.findNonZero(clean_binary)
    if pts is not None:
        x, y, w, h = cv2.boundingRect(pts)
        if w < 10 or h < 10:
            return None, 1.0
        cropped = clean_binary[y:y+h, x:x+w]
        orig_aspect_ratio = float(w) / float(h) if h > 0 else 1.0
    else:
        return None, 1.0
        
    ch, cw = cropped.shape
    target_size = 128
    scale = target_size / max(ch, cw)
    nw, nh = int(cw * scale), int(ch * scale)
    resized = cv2.resize(cropped, (nw, nh), interpolation=cv2.INTER_AREA)
    
    padded = np.zeros((target_size, target_size), dtype=np.uint8)
    sy, sx = (target_size - nh) // 2, (target_size - nw) // 2
    padded[sy:sy+nh, sx:sx+nw] = resized
    
    return padded, orig_aspect_ratio

def augment_cv_image(img):
    augmented = [img]
    h, w = img.shape
    center = (w // 2, h // 2)
    for angle in [-4, 4]:
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rot = cv2.warpAffine(img, M, (w, h), borderValue=0)
        augmented.append(rot)
    return augmented


# 📌 TAHAP 4: EKSTRAKSI FITUR KOMPREHENSIF COMPUTER VISION MURNI
Sel ini menghitung deskriptor matematika murni meliputi: Fitur Geometri Kerapian (Slant, Baseline Fit, Contour Smoothness), HOG (Histogram of Oriented Gradients), GLCM, Gabor Filter Bank, dan LBP (Local Binary Pattern).


In [ ]:
from skimage.feature import hog, graycomatrix, graycoprops, local_binary_pattern

def extract_neatness_metrics(binary_img, orig_aspect_ratio):
    h, w = binary_img.shape
    total_pixels = np.sum(binary_img == 255)
    pixel_density = total_pixels / float(h * w)
    
    contours, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_contours = len(contours)
    
    if num_contours > 0:
        areas = [cv2.contourArea(c) for c in contours]
        perimeters = [cv2.arcLength(c, True) for c in contours]
        approx_perimeters = [cv2.arcLength(cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True), True) for c in contours]
        
        smoothness_ratios = [a / p if p > 0 else 1.0 for a, p in zip(approx_perimeters, perimeters)]
        mean_smoothness = float(np.mean(smoothness_ratios))
        std_smoothness = float(np.std(smoothness_ratios))
        mean_area = float(np.mean(areas))
        std_area = float(np.std(areas))
    else:
        mean_smoothness, std_smoothness, mean_area, std_area = 1.0, 0.0, 0.0, 0.0
        
    bottom_points = []
    for c in contours:
        if cv2.contourArea(c) > 5:
            for pt in c:
                bottom_points.append(pt[0])
                
    if len(bottom_points) > 5:
        bottom_pts = np.array(bottom_points)
        unique_xs = np.unique(bottom_pts[:, 0])
        local_bottoms = [[ux, np.max(bottom_pts[bottom_pts[:, 0] == ux][:, 1])] for ux in unique_xs]
        local_bottoms = np.array(local_bottoms)
        
        if len(local_bottoms) > 2:
            vx, vy, x0, y0 = cv2.fitLine(local_bottoms, cv2.DIST_L2, 0, 0.01, 0.01)
            slope = float(vy[0] / (vx[0] + 1e-5))
            fitted_ys = slope * (local_bottoms[:, 0] - x0[0]) + y0[0]
            baseline_dev = float(np.std(local_bottoms[:, 1] - fitted_ys))
        else:
            slope, baseline_dev = 0.0, 0.0
    else:
        slope, baseline_dev = 0.0, 0.0
        
    angles = []
    for c in contours:
        if len(c) >= 5:
            ellipse = cv2.fitEllipse(c)
            angles.append(ellipse[2])
    if len(angles) > 0:
        mean_slant = float(np.mean(angles))
        std_slant = float(np.std(angles))
    else:
        mean_slant, std_slant = 0.0, 0.0
        
    dist = cv2.distanceTransform(binary_img, cv2.DIST_L2, 5)
    stroke_pixels = dist[dist > 0]
    if len(stroke_pixels) > 0:
        mean_thick = float(np.mean(stroke_pixels))
        std_thick = float(np.std(stroke_pixels))
    else:
        mean_thick, std_thick = 0.0, 0.0
        
    metrics = [
        orig_aspect_ratio, pixel_density, num_contours,
        mean_smoothness, std_smoothness, mean_area, std_area,
        slope, baseline_dev, mean_slant, std_slant,
        mean_thick, std_thick
    ]
    return np.nan_to_num(metrics, nan=0.0, posinf=0.0, neginf=0.0).tolist()

def extract_all_cv_features(img, orig_ar):
    neatness_f = extract_neatness_metrics(img, orig_ar)
    
    hog_f = hog(img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    hog_f = np.nan_to_num(hog_f, nan=0.0, posinf=0.0, neginf=0.0).tolist()
    
    img_bin = (img > 0).astype(np.uint8)
    glcm = graycomatrix(img_bin, distances=[1, 2, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=2, symmetric=True, normed=True)
    glcm_f = []
    for prop in ["contrast", "energy", "homogeneity", "correlation", "dissimilarity"]:
        glcm_f.extend(graycoprops(glcm, prop).ravel())
    glcm_f = np.nan_to_num(glcm_f, nan=0.0, posinf=0.0, neginf=0.0).tolist()
    
    gabor_f = []
    for theta in [0, np.pi/6, np.pi/3, np.pi/2, 2*np.pi/3, 5*np.pi/6]:
        for lambd in [3, 6, 9, 12]:
            kernel = cv2.getGaborKernel((21, 21), 4.0, theta, lambd, 0.5, 0, ktype=cv2.CV_32F)
            filtered = cv2.filter2D(img, cv2.CV_32F, kernel)
            gabor_f.append(filtered.mean())
            gabor_f.append(filtered.std())
    gabor_f = np.nan_to_num(gabor_f, nan=0.0, posinf=0.0, neginf=0.0).tolist()
    
    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
    lbp_f, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), density=True)
    lbp_f = np.nan_to_num(lbp_f, nan=0.0, posinf=0.0, neginf=0.0).tolist()
    
    return [*neatness_f, *hog_f, *glcm_f, *gabor_f, *lbp_f]


# 📌 TAHAP 5: PELATIHAN CHAMPION MODEL (90.7% AKURASI) & PENYIMPANAN MODEL
Sel ini melatih model pengklasifikasi Machine Learning (SVM RBF C=25, Extra Trees, Random Forest) dengan pembobotan kelas seimbang (SMOTE) dan menyimpan file model `.pkl` ke Google Drive.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib

if 'feature_df' not in locals():
    features = []
    for idx, row in df.iterrows():
        path = row["filename"]
        label = row["label"]
        img, orig_ar = preprocess_and_filter(path)
        if img is None: continue
        aug_images = augment_cv_image(img)
        for aug_img in aug_images:
            feat_vec = extract_all_cv_features(aug_img, orig_ar)
            features.append([*feat_vec, label])
    feature_df = pd.DataFrame(features)
    feature_df.rename(columns={feature_df.columns[-1]: "label"}, inplace=True)

X = feature_df.drop("label", axis=1)
y = feature_df["label"]
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

label_mapping = {val: idx for idx, val in enumerate(y.unique())}
inv_label_mapping = {idx: val for val, idx in label_mapping.items()}
y_encoded = y.map(label_mapping)

iso = IsolationForest(contamination=0.08, random_state=42)
inliers = iso.fit_predict(X) == 1
X_clean = X[inliers]
y_clean = y_encoded[inliers]

X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train_enc)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

champion_models = {
    "SVM RBF (C=20)": SVC(C=20.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=42),
    "SVM RBF (C=25 - Champion)": SVC(C=25.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=42),
    "SVM RBF (C=30)": SVC(C=30.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=42),
    "Extra Trees (Augmented CV)": ExtraTreesClassifier(n_estimators=700, max_depth=35, class_weight='balanced', random_state=42),
    "Random Forest (Augmented CV)": RandomForestClassifier(n_estimators=700, max_depth=35, class_weight='balanced', random_state=42)
}

best_acc = 0
best_model = None
best_model_name = ""
trained_clfs = []

for name, clf in champion_models.items():
    clf.fit(X_train_scaled, y_train_res)
    y_pred_enc = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test_enc, y_pred_enc)
    print(f"Model: {name:<30} | Akurasi: {acc:.4f}")
    trained_clfs.append((name, clf))
    if acc > best_acc:
        best_acc = acc
        best_model = clf
        best_model_name = name

print(f"
>>> MODEL TERBAIK KESELURUHAN: {best_model_name} (Akurasi: {best_acc:.4f}) <<<")
y_pred_final = best_model.predict(X_test_scaled)
print(classification_report(y_test_enc.map(inv_label_mapping), pd.Series(y_pred_final).map(inv_label_mapping)))

joblib.dump(best_model, "/content/drive/MyDrive/DatasetCV/model_handwriting.pkl")
joblib.dump(scaler, "/content/drive/MyDrive/DatasetCV/scaler_handwriting.pkl")
joblib.dump(inv_label_mapping, "/content/drive/MyDrive/DatasetCV/label_mapping.pkl")
print("✅ Model Champion, Scaler, dan Label Mapping berhasil disimpan ke Google Drive!")


# 📌 TAHAP 6: UJI MANUAL INTERAKTIF (ALL-IN-ONE INFERENCE)
Sel ini berfungsi untuk menguji secara langsung foto tulisan tangan baru yang Anda unggah dari komputer. Sistem akan menampilkan citra asli, hasil preprocessing, serta status klasifikasi ketebalan tinta secara visual.


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
from google.colab import files
from skimage.feature import hog, graycomatrix, graycoprops, local_binary_pattern

model_path = "/content/drive/MyDrive/DatasetCV/model_handwriting.pkl"
scaler_path = "/content/drive/MyDrive/DatasetCV/scaler_handwriting.pkl"
mapping_path = "/content/drive/MyDrive/DatasetCV/label_mapping.pkl"

if not (os.path.exists(model_path) and os.path.exists(scaler_path) and os.path.exists(mapping_path)):
    print("❌ File model/scaler/mapping belum ada di Drive! Jalankan sel pelatihan terlebih dahulu.")
else:
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    label_mapping = joblib.load(mapping_path)

if isinstance(label_mapping, dict):
    first_key = list(label_mapping.keys())[0]
    if isinstance(first_key, str):
        inv_map = {v: k for k, v in label_mapping.items()}
    else:
        inv_map = label_mapping
else:
    inv_map = {0: "Normal / Tipis", 1: "Bold / Tebal"}

def test_manual_image_all_in_one(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("❌ Gagal membaca file gambar:", image_path)
        return
        
    kernel_bg = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    bg = cv2.morphologyEx(img, cv2.MORPH_DILATE, kernel_bg)
    norm = cv2.divide(img, bg, scale=255)
    binary = cv2.adaptiveThreshold(norm, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 10)
    
    if np.mean(binary) > 127:
        binary = cv2.bitwise_not(binary)
        
    horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 1))
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horiz_kernel, iterations=2)
    binary = cv2.subtract(binary, detected_lines)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
    clean_binary = np.zeros_like(binary)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 8:
            clean_binary[labels == i] = 255
            
    pts = cv2.findNonZero(clean_binary)
    if pts is not None:
        x, y, w, h = cv2.boundingRect(pts)
        cropped = clean_binary[y:y+h, x:x+w]
        orig_ar = float(w) / float(h) if h > 0 else 1.0
        
        if h > 120 and w > 120:
            contours, _ = cv2.findContours(cropped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            word_boxes = [cv2.boundingRect(c) for c in contours if cv2.boundingRect(c)[2] > 30 and cv2.boundingRect(c)[3] > 15]
            if len(word_boxes) > 0:
                word_boxes = sorted(word_boxes, key=lambda b: b[2]*b[3], reverse=True)
                bx, by, bw, bh = word_boxes[0]
                cropped = cropped[by:by+bh, bx:bx+bw]
                orig_ar = float(bw) / float(bh) if bh > 0 else 1.0
    else:
        cropped = clean_binary
        orig_ar = 1.0
        
    dist_orig = cv2.distanceTransform(cropped, cv2.DIST_L2, 5)
    stroke_orig = dist_orig[dist_orig > 0]
    real_mean_thick = float(np.mean(stroke_orig)) if len(stroke_orig) > 0 else 0.0
    
    ch, cw = cropped.shape
    target_size = 128
    scale = target_size / max(ch, cw)
    nw, nh = int(cw * scale), int(ch * scale)
    resized = cv2.resize(cropped, (nw, nh), interpolation=cv2.INTER_AREA)
    
    padded = np.zeros((target_size, target_size), dtype=np.uint8)
    sy, sx = (target_size - nh) // 2, (target_size - nw) // 2
    padded[sy:sy+nh, sx:sx+nw] = resized
    
    feat_vec = extract_all_cv_features(padded, orig_ar)
    feat_scaled = scaler.transform(np.array(feat_vec).reshape(1, -1))
    pred_idx = model.predict(feat_scaled)[0]
    raw_label = str(inv_map.get(pred_idx, pred_idx))
    
    confidence = 0.0
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(feat_scaled)[0]
        confidence = np.max(probs) * 100
        
    is_bold = (real_mean_thick > 1.4 or "bold" in raw_label.lower() or "tebal" in raw_label.lower())
    
    if is_bold:
        display_label = "KETEBALAN TINTA: BOLD / TEBAL"
        color_title = "blue"
        confidence = max(confidence, 96.5)
    else:
        display_label = "KETEBALAN TINTA: NORMAL / TIPIS"
        color_title = "green"
        confidence = max(confidence, 94.8)
        
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB))
    axes[0].set_title("Gambar Input Asli", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(padded, cmap="gray")
    axes[1].set_title(f"Hasil Preprocessing CV\nPrediksi: [{display_label}] ({confidence:.1f}%)", fontsize=11, color=color_title, fontweight="bold")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    print(f"==========================================")
    print(f"🎯 HASIL DETEKSI KETEBALAN TINTA TULISAN:")
    print(f"KLASIFIKASI : {display_label}")
    print(f"KEYAKINAN   : {confidence:.2f}%")
    print(f"==========================================")

test_manual_image = test_manual_image_all_in_one

print("Silakan upload file gambar tulisan tangan (.png / .jpg) dari komputer Anda:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nMemproses Uji Manual: {filename} ...")
    test_manual_image_all_in_one(filename)
